In [120]:
import requests
import pandas as pd
import time

In [2]:
#Config
owner = "pandas-dev"
repo = "pandas"

base_url = f"https://api.github.com/repos/{owner}/{repo}/issues"

params = {
    "state": "all",
    "per_page": 100,
    "page": 1
}

all_issues = []

In [3]:
#Download issues
for page in range(1, 6):
    
    params["page"] = page
    
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print("Request failed:", response.status_code)
        break
        
    data = response.json()
    
    if len(data) == 0:
        break
        
    all_issues.extend(data)
    
    time.sleep(1)

print("Downloaded issues:", len(all_issues))

Downloaded issues: 500


In [4]:
#Issues to pandas tables
issue_rows = []
skipped_prs = 0

for issue in all_issues:
    
    if "pull_request" in issue:
        skipped_prs += 1
        continue
    
    row = {
        "issue_id": issue["number"],
        "title": issue["title"],
        "body": issue["body"],
        "author": issue["user"]["login"] if issue["user"] else None,
        "created_at": issue["created_at"],
        "state": issue["state"],
        "comments_count": issue["comments"]
    }
    
    issue_rows.append(row)

issues_df = pd.DataFrame(issue_rows)

print(f"Skipped {skipped_prs} pull reqs")
print("Issues:", len(issues_df))
issues_df.head()

Skipped 403 pull reqs
Issues: 97


,issue_id,title,body,author,created_at,state,comments_count
0,64460,DEPR: deprecate set_eng_float_format,Claude wrote this at my request.\n\n## Proposa...,jbrockmendel,2026-03-07T22:42:00Z,open,0
1,64458,DOC: Fix incorrect Sphinx cross references in ...,The top level `doc/` directory contains [reStr...,mroeschke,2026-03-07T19:42:11Z,open,0
2,64454,Add ARM64 (AArch64) benchmark coverage (ASV ru...,### Research\n\n- [x] I have searched the [[pa...,113xiaoji,2026-03-07T16:24:36Z,open,1
3,64450,API: Add reduction methods to ExtensionArray s...,The subclasses of `ExtensionArray` implemented...,rhshadrach,2026-03-07T12:42:02Z,open,0
4,64446,DOC: Public APIs missing from the API referenc...,### Pandas version checks\n\n- [x] I have chec...,tuhinsharma121,2026-03-07T07:05:59Z,closed,0


In [5]:
all_comments = []

In [6]:
#Download comments
for issue_number in issues_df["issue_id"].head(100):
    
    url = f"https://api.github.com/repos/{owner}/{repo}/issues/{issue_number}/comments"
    
    response = requests.get(url)
    
    if response.status_code != 200:
        continue
        
    comments = response.json()
    
    for c in comments:
        
        row = {
            "comment_id": c["id"],
            "issue_id": issue_number,
            "author": c["user"]["login"] if c["user"] else None,
            "body": c["body"],
            "created_at": c["created_at"]
        }
        
        all_comments.append(row)
        
    time.sleep(0.5)

comments_df = pd.DataFrame(all_comments)

print("Comments:", len(comments_df))
comments_df.head()

Comments: 52


,comment_id,issue_id,author,body,created_at
0,4017579083,64454,rhshadrach,Thanks for you're interest here! cc @pandas-de...,2026-03-07T22:48:26Z
1,4013843349,64428,Desel72,"Hi maintainer team, Can I work on this issue?",2026-03-06T20:00:44Z
2,4015950736,64428,An1k4et,"hi, I have worked on this issue and implemente...",2026-03-07T08:24:58Z
3,4012776005,64426,Dr-Irv,closing in favor of #64419,2026-03-06T16:37:07Z
4,4011011092,64419,smith558,blocking https://github.com/pandas-dev/pandas/...,2026-03-06T10:51:02Z


In [121]:
#Issue + Comments -> Artifacts
issue_text_rows = []

for _, row in issues_df.iterrows():
    
    text = (row["title"] or "") + " " + (row["body"] or "")
    
    issue_text_rows.append({
        "artifact_id": f"issue_{row['issue_id']}",
        "issue_id": row["issue_id"],
        "author": row["author"],
        "text": text,
        "timestamp": row["created_at"],
        "type": "issue"
    })

issues_text_df = pd.DataFrame(issue_text_rows)

In [122]:
comment_text_rows = []

for _, row in comments_df.iterrows():
    
    comment_text_rows.append({
        "artifact_id": f"comment_{row['comment_id']}",
        "issue_id": row["issue_id"],
        "author": row["author"],
        "text": row["body"],
        "timestamp": row["created_at"],
        "type": "comment"
    })

comments_text_df = pd.DataFrame(comment_text_rows)

In [123]:
import re

def clean_text(text):
    
    if pd.isna(text):
        return ""
    
    text = text.lower()
    
    text = re.sub(r"http\S+", " ", text)
    
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

#Future improvement: advanced preprocessing
#Possible upgrades:
#- remove code blocks
#- remove quoted email text
#- detect sentences
#- detect named entities
#These would improve extraction accuracy.

In [124]:
artifacts_df = pd.concat([issues_text_df, comments_text_df], ignore_index=True)
artifacts_df["clean_text"] = artifacts_df["text"].apply(clean_text)

print("Artifacts:", len(artifacts_df))
artifacts_df.head()

Artifacts: 149


,artifact_id,issue_id,author,text,timestamp,type,clean_text
0,issue_64460,64460,jbrockmendel,DEPR: deprecate set_eng_float_format Claude wr...,2026-03-07T22:42:00Z,issue,depr: deprecate set_eng_float_format claude wr...
1,issue_64458,64458,mroeschke,DOC: Fix incorrect Sphinx cross references in ...,2026-03-07T19:42:11Z,issue,doc: fix incorrect sphinx cross references in ...
2,issue_64454,64454,113xiaoji,Add ARM64 (AArch64) benchmark coverage (ASV ru...,2026-03-07T16:24:36Z,issue,add arm64 (aarch64) benchmark coverage (asv ru...
3,issue_64450,64450,rhshadrach,API: Add reduction methods to ExtensionArray s...,2026-03-07T12:42:02Z,issue,api: add reduction methods to extensionarray s...
4,issue_64446,64446,tuhinsharma121,DOC: Public APIs missing from the API referenc...,2026-03-07T07:05:59Z,issue,doc: public apis missing from the api referenc...


In [125]:
issue_ref_pattern = r"#(\d+)"

duplicate_patterns = [
    r"duplicate of #(\d+)",
    r"dup of #(\d+)",
    r"dup #(\d+)"
]

proposal_patterns = [
    r"\bshould\b",
    r"\bcould\b",
    r"\bmaybe\b",
    r"\bsuggest\b",
    r"\bconsider\b",
    r"\bproposal\b",
    r"\bit would be nice\b",
    r"\bit would be useful\b",
    r"\bwe might\b",
    r"\bwe should\b",
    r"^(add|allow|support)\b"
]

In [126]:
# RULE BASED EXTRACTION WILL GO HERE

# FUTURE UPGRADE
# Replace rule-based extraction with LLM extraction
#
# The LLM would receive the text and return structured JSON:
#
# {
#   subject:
#   relation:
#   object:
# }
#
# This keeps the rest of the pipeline unchanged.

def extract_evidence(text, max_len=200):
    """
    Extract the first meaningful sentence instead of
    blindly truncating the first 200 characters.
    """
    sentences = re.split(r"[.!?\n]", text)

    for s in sentences:
        s = s.strip()
        if len(s) > 20:
            return s[:max_len]

    return text[:max_len]
    
def extract_claims(text, artifact_id, issue_id, author, timestamp):
    
    claims = []
    
    if text is None or len(text.strip()) == 0: return claims
    text_lower = text.lower()
    evidence = extract_evidence(text)
    
    
    # --- Rule 1: Issue references (#123) ---
    
    refs = re.findall(issue_ref_pattern, text_lower)

    duplicate_targets = set()
    for pattern in duplicate_patterns:
        duplicate_targets.update(re.findall(pattern, text_lower))    

    for ref in refs:
    
        if ref in duplicate_targets: continue
        
        claim = {
            "subject": f"issue_{issue_id}",
            "relation": "RELATES_TO",
            "object": f"issue_{ref}",
            "artifact_id": artifact_id,
            "issue_id": issue_id,
            "author": author,
            "timestamp": timestamp,
            "evidence": evidence        
            }
        
        claims.append(claim)
    
    
    # --- Rule 2: Duplicate detection ---
    
    for pattern in duplicate_patterns:
        
        matches = re.findall(pattern, text_lower)
        
        for ref in matches:
            
            claim = {
                "subject": f"issue_{issue_id}",
                "relation": "DUPLICATE_OF",
                "object": f"issue_{ref}",
                "artifact_id": artifact_id,
                "issue_id": issue_id,
                "author": author,
                "timestamp": timestamp,
                "evidence": text[:200]
            }
            
            claims.append(claim)
    
    
    # --- Rule 3: Proposal detection ---
    
    for pattern in proposal_patterns:

        if re.search(pattern, text_lower):
            
            claim = {
                "subject": author,
                "relation": "PROPOSES",
                "object": f"issue_{issue_id}",
                "artifact_id": artifact_id,
                "issue_id": issue_id,
                "author": author,
                "timestamp": timestamp,
                "evidence": text[:200]
            }
            
            claims.append(claim)
            
            break
    
    
    return claims

In [127]:
all_claims = []

for _, row in artifacts_df.iterrows():
        
    claims = extract_claims(
        row["text"],
        row["artifact_id"],
        row["issue_id"],
        row["author"],
        row["timestamp"]
    )
    
    all_claims.extend(claims)

print("Total extracted claims:", len(all_claims))

Total extracted claims: 98


In [130]:
claims_df = pd.DataFrame(all_claims)

claims_df.sort_values(
    ["issue_id", "relation"]
).to_csv("outputs/claims_df.csv", index=False)

In [131]:
#Sanity check
claims_df["relation"].value_counts()

relation
PROPOSES      59
RELATES_TO    39
Name: count, dtype: int64

In [132]:
claims_df.sample(10)[["evidence", "relation"]]

,evidence,relation
63,item() query no longer possible since unique()...,RELATES_TO
17,BUG: `eval` not working with string containmen...,RELATES_TO
88,@MarcoGorelli thanks for the report! It seems ...,PROPOSES
42,API: PeriodIndex(dt64_data) depends on contain...,PROPOSES
29,ENH: add Expression for index ### Feature Type...,PROPOSES
49,BUG: test_assignment_not_inplace incorrectly x...,RELATES_TO
89,"In Localizer class, pytz caches transition arr...",PROPOSES
20,PERF: datetime index getters functions are 10 ...,RELATES_TO
55,BUG: Inconsistent number conversion with leadi...,PROPOSES
0,DEPR: deprecate set_eng_float_format Claude wr...,RELATES_TO


In [133]:
claims_df.groupby("evidence").size().sort_values(ascending=False).head(5)

evidence
DEPR: Period "freq" -> "unit" Discussed in last week's meeting and before that in #61897, but before pulling the trigger on the rest of this (#64155 and more) I decided to write up The Whole Thing:    6
@rhshadrach This issue seems to be spam                                                                                                                                                                  4
DEPR: deprecate set_eng_float_format Claude wrote this at my request                                                                                                                                     4
DEPR: float_precision in read_csv We have "high" (the default), "legacy", and "round_trip"                                                                                                               2
> Thanks for the report, this was due to [#62437](https://github                                                                                                                   

In [134]:
print("Artifacts:", len(artifacts_df))
print("Claims:", len(claims_df))
print("Unique evidence:", claims_df["evidence"].nunique())
print("Duplicate evidence:", len(claims_df) - claims_df["evidence"].nunique())

Artifacts: 149
Claims: 98
Unique evidence: 86
Duplicate evidence: 12


In [135]:
conversation_patterns = [
    r"^i agree",
    r"^thanks",
    r"^thank you",
    r"^this seems",
    r"^i think",
    r"^looks like",
]


#Clean claim
def clean_claim_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    # remove common github issue prefixes
    text = re.sub(r"^(bug|doc|docs|enh|feature|depr|api|perf|refactor)\s*:\s*", "", text)
    
    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    #get rid of codeblocks
    text = re.sub(r"`[^`]*`", "CODE_TOKEN", text)
    # remove markdown headers
    text = re.sub(r"^#+\s.*", "", text)

    # remove quoted replies
    text = re.sub(r"^>\s.*", "", text)

    # remove template sections
    text = re.sub(r"###\s.*", "", text)

    for p in conversation_patterns:
        if re.search(p, text):
            return ""
    
    return text


claims_df["clean_evidence"] = claims_df["evidence"].apply(clean_claim_text)


# drop very short claims
claims_df = claims_df[claims_df["clean_evidence"].str.len() > 20]
claims_df = claims_df.drop_duplicates(
    subset=["issue_id", "clean_evidence"]
)


print("Claims after cleaning:", len(claims_df))


Claims after cleaning: 78


In [136]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [137]:
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = claims_df["clean_evidence"].tolist()

X = model.encode(
    sentences,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [138]:
similarity_matrix = cosine_similarity(X)

In [140]:
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict

distance_matrix = 1 - similarity_matrix

clustering = AgglomerativeClustering(
    metric="precomputed",
    linkage="average",
    distance_threshold=0.25,
    n_clusters=None
)

labels = clustering.fit_predict(distance_matrix)

cluster_dict = defaultdict(list)

for i, label in enumerate(labels):
    cluster_dict[label].append(i)

clusters = list(cluster_dict.values())

In [141]:
pd.Series([len(c) for c in clusters]).value_counts().sort_index()

1    64
2     5
4     1
Name: count, dtype: int64

In [142]:
len(clusters), len(claims_df)

(70, 78)

In [143]:
import numpy as np
canonical_claims = []

for cluster in clusters:

    rows = claims_df.iloc[cluster]

    # slice similarity matrix for just this cluster
    cluster_sim = similarity_matrix[np.ix_(cluster, cluster)].copy()

    # ignore self-similarity
    np.fill_diagonal(cluster_sim, 0)

    # average similarity to other claims in cluster
    avg_sim = cluster_sim.mean(axis=1)

    canonical_position = avg_sim.argmax()

    canonical_text = rows.iloc[canonical_position]["clean_evidence"]

    # top representative evidence examples
    evidence_positions = avg_sim.argsort()[::-1][:3]
    evidence_examples = rows.iloc[evidence_positions]["evidence"].tolist()

    canonical_timestamp = rows["timestamp"].min()

    canonical = {
        "canonical_claim": canonical_text,
        "support_count": len(cluster),
        "relations": list(rows["relation"].unique()),
        "issue_ids": list(rows["issue_id"].unique()),
        "timestamp": canonical_timestamp,
        "evidence_examples": evidence_examples
    }

    canonical_claims.append(canonical)

dedup_df = pd.DataFrame(canonical_claims)

In [145]:
print("Original claims:", len(claims_df))
print("Canonical claims:", len(dedup_df))

dedup_df.sort_values("support_count", ascending=False).to_csv("outputs/dedup_df.csv", index=False)
dedup_df["support_count"].describe()

Original claims: 78
Canonical claims: 70


count    70.000000
mean      1.114286
std       0.435486
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       4.000000
Name: support_count, dtype: float64

In [146]:
def inspect_cluster(cluster_id, clusters, claims_df):

    cluster = clusters[cluster_id]

    rows = claims_df.iloc[cluster]

    print(f"\nCluster {cluster_id}")
    print(f"Size: {len(cluster)}\n")

    for r in rows["clean_evidence"]:
        print("-", r)

In [147]:
for i, c in enumerate(clusters):
    if len(c) > 1:
        print(i, len(c))

0 2
6 2
8 4
14 2
41 2
55 2


In [148]:
inspect_cluster(42, clusters, claims_df)


Cluster 42
Size: 1

- item() query no longer possible since unique() method no longer returns array 


In [149]:
sims = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

np.percentile(sims, [50, 75, 90, 95, 99])

array([0.12209698, 0.22114232, 0.3508835 , 0.4583241 , 0.65012962])

In [150]:
dedup_df["claim_id"] = [f"C{i:04d}" for i in range(len(dedup_df))]
dedup_df.head()

,canonical_claim,support_count,relations,issue_ids,timestamp,evidence_examples,claim_id
0,deprecate set_eng_float_format claude wrote th...,2,"[RELATES_TO, PROPOSES]",[64460],2026-03-07T22:42:00Z,[DEPR: deprecate set_eng_float_format Claude w...,C0000
1,fix incorrect sphinx cross references in older...,1,[PROPOSES],[64458],2026-03-07T19:42:11Z,[DOC: Fix incorrect Sphinx cross references in...,C0001
2,add arm64 (aarch64) benchmark coverage (asv ru...,1,[PROPOSES],[64454],2026-03-07T16:24:36Z,[Add ARM64 (AArch64) benchmark coverage (ASV r...,C0002
3,add reduction methods to extensionarray subcla...,1,[PROPOSES],[64450],2026-03-07T12:42:02Z,[API: Add reduction methods to ExtensionArray ...,C0003
4,public apis missing from the api reference doc...,1,[PROPOSES],[64446],2026-03-07T07:05:59Z,[DOC: Public APIs missing from the API referen...,C0004


In [151]:
evidence_rows = []

for _, row in dedup_df.iterrows():

    for ev in row["evidence_examples"]:
        evidence_rows.append({
            "claim_id": row["claim_id"],
            "evidence_text": ev
        })

evidence_df = pd.DataFrame(evidence_rows)

In [152]:
artifact_ids = set()

for issues in dedup_df["issue_ids"]:
    artifact_ids.update(issues)

artifacts_df = pd.DataFrame({
    "artifact_id": list(artifact_ids)
})

In [153]:
claim_artifact_edges = []

for _, row in dedup_df.iterrows():

    for issue in row["issue_ids"]:
        claim_artifact_edges.append({
            "claim_id": row["claim_id"],
            "artifact_id": issue
        })

claim_artifact_df = pd.DataFrame(claim_artifact_edges)

In [154]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [155]:
def extract_entities(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents]

dedup_df["entities"] = dedup_df["canonical_claim"].apply(extract_entities)

entities = dedup_df["entities"].explode().dropna().unique()

entities_df = pd.DataFrame({
    "entity_id": [f"E{i:04d}" for i in range(len(entities))],
    "entity_name": entities
})

In [156]:
entities_df["entity_name"].value_counts().head(20)

entity_name
CODE_TOKEN                              1
restructuredtext](https://www.sphinx    1
asv runner                              1
kunpeng                                 1
pandas                                  1
10                                      1
zoneinfo vs pytz                        1
pd.na                                   1
1                                       1
3                                       1
5                                       1
pandas.array                            1
64264                                   1
first                                   1
dti = pd.datetimeindex(data             1
assert dti.freq                         1
404                                     1
zeros                                   1
to_numeric                              1
last week's                             1
Name: count, dtype: int64

In [157]:
metrics = {
    "raw_claims": len(claims_df),
    "canonical_claims": len(dedup_df),
    "dedup_ratio": len(dedup_df) / len(claims_df),
    "entities": len(entities_df)
}

metrics

{'raw_claims': 78,
 'canonical_claims': 70,
 'dedup_ratio': 0.8974358974358975,
 'entities': 30}

In [158]:
entity_lookup = dict(zip(entities_df["entity_name"], entities_df["entity_id"]))

edges = []

for _, row in dedup_df.iterrows():

    for ent in row["entities"]:
        edges.append({
            "claim_id": row["claim_id"],
            "entity_id": entity_lookup[ent]
        })

claim_entity_df = pd.DataFrame(edges)

In [159]:
claim_entity_df.to_csv("outputs/claim_entity_edges.csv", index=False)

In [160]:
canonical_embeddings = model.encode(
    dedup_df["canonical_claim"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True
)
np.save("outputs/embeddings.npy", canonical_embeddings)

In [161]:
def retrieve_context(question, top_k=5):

    # embed the question
    q_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # compute similarity
    sims = cosine_similarity(q_embedding, canonical_embeddings)[0]

    # get top claims
    top_indices = sims.argsort()[::-1][:top_k]

    results = []

    for idx in top_indices:

        claim_row = dedup_df.iloc[idx]

        results.append({
            "claim": claim_row["canonical_claim"],
            "score": float(sims[idx]),
            "support_count": claim_row["support_count"],
            "entities": claim_row.get("entities", []),
            "artifacts": claim_row["issue_ids"],
            "evidence": claim_row["evidence_examples"]
        })

    return results

In [162]:
def print_context(results):

    for i, r in enumerate(results, 1):

        print(f"\nResult {i}")
        print("Claim:", r["claim"])
        print("Similarity:", round(r["score"], 3))
        print("Support:", r["support_count"])
        print("Entities:", r["entities"])
        print("Artifacts:", r["artifacts"])

        print("\nEvidence:")
        for e in r["evidence"]:
            print("-", e)

In [163]:
results = retrieve_context("What's wrong with Pandas?")
print_context(results)


Result 1
Claim: wrong source link for `pandas
Similarity: 0.587
Support: 1
Entities: []
Artifacts: [np.int64(64224)]

Evidence:
- DOC: wrong source link for `pandas

Result 2
Claim: pandas read_sas not working anymore 
Similarity: 0.574
Support: 1
Entities: ['pandas']
Artifacts: [np.int64(64055)]

Evidence:
- BUG: pandas read_sas not working anymore ### Pandas version checks

- [x] I have checked that this issue has not already been reported.

- [x] I have confirmed this bug exists on the [latest version](

Result 3
Claim: suggestion: link a rag / llm debugging checklist (wfgy problem map) in the docs hi pandas team, thank you for maintaining such a foundational library. pandas is often the first stop for people who ar
Similarity: 0.545
Support: 1
Entities: ['pandas', 'first']
Artifacts: [np.int64(64263)]

Evidence:
- Suggestion: link a RAG / LLM debugging checklist (WFGY Problem Map) in the docs Hi pandas team,

thank you for maintaining such a foundational library. pandas is often t

In [164]:
import networkx as nx

G = nx.Graph()

# add entity nodes
for _, row in entities_df.iterrows():
    G.add_node(
        row["entity_id"],
        label=row["entity_name"],
        type="entity"
    )

# add claim nodes
for _, row in dedup_df.iterrows():
    G.add_node(
        row["claim_id"],
        label=row["canonical_claim"],
        type="claim",
        support=row["support_count"]
    )

# add edges between claims and entities
for _, row in claim_entity_df.iterrows():
    G.add_edge(
        row["entity_id"],
        row["claim_id"]
    )

In [173]:
from pyvis.network import Network

net = Network(
    height="700px",
    width="100%",
    notebook=True
)

net.from_nx(G)

for node in net.nodes:
    
    if G.nodes[node["id"]]["type"] == "entity":
        node["color"] = "#4C72B0"
        node["shape"] = "dot"
        node["size"] = 20
    
    elif G.nodes[node["id"]]["type"] == "claim":
        node["color"] = "#DD8452"
        node["shape"] = "box"
        node["size"] = 15

for node in net.nodes:
    
    if node["id"].startswith("C"):
        
        row = dedup_df[dedup_df["claim_id"] == node["id"]].iloc[0]
        
        tooltip = "<br>".join(row["evidence_examples"])
        
        node["title"] = f"""
        <b>Claim:</b> {row["canonical_claim"]}<br>
        <b>Support:</b> {row["support_count"]}<br><br>
        <b>Evidence:</b><br>{tooltip}
        """

In [174]:
net.set_options('{"physics":{"enabled":true,"stabilization":{"enabled":true,"iterations":1000,"fit":true},"barnesHut":{"gravitationalConstant":-30000,"centralGravity":0.1,"springLength":250,"springConstant":0.05,"damping":1}},"interaction":{"hover":true,"tooltipDelay":100,"navigationButtons":true}}')
net.save_graph("visualization/graph.html")

In [175]:
nx.write_graphml(G, "outputs/memory_graph.graphml")

In [172]:
import json

def convert_numpy(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def save_context_pack(question):

    results = retrieve_context(question)

    pack = {
        "question": question,
        "retrieved_claims": results
    }

    filename = question.replace(" ", "_")[:40] + ".json"

    with open(f"retrieval_examples/{filename}", "w") as f:
        json.dump(pack, f, indent=2, default=convert_numpy)

save_context_pack("What's wrong with Pandas")
save_context_pack("What is the future of Pandas")
save_context_pack("Numpy won't work")